In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os

# 1. SETTINGS & WORD LIST
DATA_PATH = os.path.join('My_VitalSign_Data') 
# Tam 30 Kelimelik Devasa Medikal & Sosyal Veri Setimiz
actions = np.array([
    'help', 'doctor', 'emergency', 
    'pain', 'medicine', 'hospital', 'blood', 'breathe',
    'head', 'heart', 'arm', 'leg', 'allergy', 'bad',
    'fever', 'cough', 'dizzy', 'nausea',
    'today', 'yesterday', 'much', 'little',
    'diabetic', 'surgery', 'accident',
    'yes', 'no', 'hello', 'thanks', 'how-are-you'
])
sequence_length = 30 # Each video will last 30 frames

# --- INTERACTIVE INTERFACES (Kullanıcı Seçim Alanı) ---
print("--- VITAL SIGN DATA COLLECTOR CONTROL PANEL ---")
print("Available Words:")
for i, action in enumerate(actions):
    print(f"[{i}] {action}")
print("-----------------------------------------------")

# Kullanıcıya hangi kelimeyi çekmek istediğini soruyoruz
word_index = int(input(f"Enter the number of the word you want to record (0-{len(actions)-1}): "))
selected_action = actions[word_index]

# Kullanıcıya kaç video çekeceğini ve kaçıncı videodan başlayacağını soruyoruz
# (Böylece eski videoların üstüne ekleme yaparken eski veriler silinmez!)
start_sequence = int(input("Which video number do you want to START with? (e.g., 0 for new, 10 if you already have 10): "))
number_of_videos = int(input("How many videos do you want to record in this session?: "))

end_sequence = start_sequence + number_of_videos

# Seçilen kelime için klasörleri otomatik kontrol et ve oluştur
for sequence in range(start_sequence, end_sequence):
    try: 
        os.makedirs(os.path.join(DATA_PATH, selected_action, str(sequence)))
    except:
        pass

# 2. MEDIAPIPE SETUP
mp_holistic = mp.solutions.holistic 
mp_drawing = mp.solutions.drawing_utils 

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False                  
    results = model.process(image)                 
    image.flags.writeable = True                   
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) 
    return image, results

def extract_landmarks(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    
    lh_xy = lh.reshape(-1, 3)[:, :2].flatten() 
    rh_xy = rh.reshape(-1, 3)[:, :2].flatten() 
    
    face = np.array([[res.x, res.y] for res in results.face_landmarks.landmark[:12]]).flatten() if results.face_landmarks else np.zeros(12*2)
    return np.concatenate([lh_xy, rh_xy, face])

# 3. RECORDING LOOP
print(f"\n[INFO] Starting camera for word: '{selected_action.upper()}'")
print(f"[INFO] Recording videos from {start_sequence} to {end_sequence-1}")

cap = cv2.VideoCapture(0)
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    
    # Sadece seçtiğimiz o tek kelime için döngü dönüyor
    for sequence in range(start_sequence, end_sequence):
        for frame_num in range(sequence_length):
            ret, frame = cap.read()
            image, results = mediapipe_detection(frame, holistic)
            
            # DRAWING LANDMARKS
            if results.face_landmarks:
                mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS, 
                                         mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                                         mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
            if results.left_hand_landmarks:
                mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

            # Visual feedback on screen
            if frame_num == 0: 
                cv2.putText(image, 'GET READY...', (120,200), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,255, 0), 4, cv2.LINE_AA)
                cv2.putText(image, f'Word: {selected_action.upper()} | Video: {sequence}', (15,30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                cv2.imshow('VitalSign Data Collector', image)
                cv2.waitKey(4000) # 4 seconds buffer time to prepare
            else: 
                cv2.putText(image, f'Word: {selected_action.upper()} | Video: {sequence}', (15,30), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
                cv2.imshow('VitalSign Data Collector', image)

            # Extract and save landmarks
            keypoints = extract_landmarks(results)
            npy_path = os.path.join(DATA_PATH, selected_action, str(sequence), str(frame_num))
            np.save(npy_path, keypoints)

            # Press 'q' to close early
            if cv2.waitKey(10) & 0xFF == ord('q'):
                break
                
    cap.release()
    cv2.destroyAllWindows()
    print(f"\n[SUCCESS] Session completed for '{selected_action.upper()}'!")

--- VITAL SIGN DATA COLLECTOR CONTROL PANEL ---
Available Words:
[0] help
[1] doctor
[2] emergency
[3] pain
[4] medicine
[5] hospital
[6] blood
[7] breathe
[8] head
[9] heart
[10] arm
[11] leg
[12] allergy
[13] bad
[14] fever
[15] cough
[16] dizzy
[17] nausea
[18] today
[19] yesterday
[20] much
[21] little
[22] diabetic
[23] surgery
[24] accident
[25] yes
[26] no
[27] hello
[28] thanks
[29] how-are-you
-----------------------------------------------
